# pLM4CPP-XAI — Predict and Interpret New Peptide Sequences

**Recommended notebook for most users. No retraining is required.**

### Input
Upload a FASTA file or a CSV file containing `sequence_id` and `sequence`.

### Output
- four PLM-specific CPP probabilities and classifications
- mean and median ensemble probabilities
- residue-level Attention, Gradient × Input, and Integrated Gradients
- redundancy-adjusted four-PLM consensus importance
- top-20% consensus hotspots
- cross-PLM hotspot-support count
- downloadable CSV files and residue-level XAI heatmaps

### Run
In Colab choose **Runtime → Change runtime type → T4 GPU**, then click **Runtime → Run all**.


In [ ]:
!pip -q install fair-esm==2.0.0 transformers sentencepiece accelerate tensorflow==2.20.0 pandas scipy matplotlib


In [ ]:
from pathlib import Path
import subprocess, shutil

REPO_URL = "https://github.com/drkumarnandan/pLM4CPP-XAI.git"
REPO_DIR = Path("/content/pLM4CPP-XAI")

if not (REPO_DIR/"models").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git","clone","--depth","1",REPO_URL,str(REPO_DIR)],check=True)

print("Repository ready:", REPO_DIR)


In [ ]:
from google.colab import files
import json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import tensorflow as tf

MAX_LEN = 61
IG_STEPS = 32
MODEL_NAMES = ["ESM2_320","ESM2_640","ESM2_1280","ProtT5"]

MODEL_CONFIGS = {
    "ESM2_320":  {"type":"esm","loader":"esm2_t6_8M_UR50D","layer":6,"dim":320,"batch":64},
    "ESM2_640":  {"type":"esm","loader":"esm2_t30_150M_UR50D","layer":30,"dim":640,"batch":24},
    "ESM2_1280": {"type":"esm","loader":"esm2_t33_650M_UR50D","layer":33,"dim":1280,"batch":8},
    "ProtT5":    {"type":"prott5","loader":"Rostlab/prot_t5_xl_uniref50","dim":1024,"batch":8},
}

MODEL_ROOT = REPO_DIR/"models"
OUTPUT = Path("/content/pLM4CPP_XAI_results")
OUTPUT.mkdir(parents=True, exist_ok=True)

required=[]
for m in MODEL_NAMES:
    required += [
        MODEL_ROOT/m/"final_attention_classifier.keras",
        MODEL_ROOT/m/"selected_threshold.json",
    ]
missing=[p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing trained-model files:\n" + "\n".join(map(str,missing)))

print("✓ Exact trained classifiers found.")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
uploaded = files.upload()
input_name = next(iter(uploaded))
input_path = Path("/content")/input_name
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def read_fasta(path):
    records=[]; sid=None; seq=[]
    for raw in path.read_text().splitlines():
        line=raw.strip()
        if not line:
            continue
        if line.startswith(">"):
            if sid is not None:
                records.append((sid,"".join(seq)))
            sid=line[1:].strip().split()[0] or f"seq_{len(records)+1}"
            seq=[]
        else:
            seq.append(line)
    if sid is not None:
        records.append((sid,"".join(seq)))
    return pd.DataFrame(records,columns=["sequence_id","sequence"])

if input_path.suffix.lower() in [".fa",".fasta",".faa"]:
    user_df=read_fasta(input_path)
else:
    user_df=pd.read_csv(input_path)
    if "sequence" not in user_df.columns:
        raise ValueError("CSV must contain a 'sequence' column.")
    if "sequence_id" not in user_df.columns:
        user_df.insert(0,"sequence_id",[f"seq_{i+1}" for i in range(len(user_df))])

user_df["sequence_id"]=user_df["sequence_id"].astype(str)
user_df["sequence"]=user_df["sequence"].astype(str).str.upper().str.replace(r"[^A-Z]","",regex=True)

bad=[]
for row in user_df.itertuples():
    invalid=sorted(set(row.sequence)-STANDARD_AA)
    if not row.sequence or len(row.sequence)>MAX_LEN or invalid:
        bad.append((row.sequence_id,len(row.sequence),invalid))
if bad:
    print("Invalid sequence(s):",bad)
    raise ValueError(f"Use only the 20 standard amino acids and sequences ≤ {MAX_LEN} residues.")
if user_df["sequence_id"].duplicated().any():
    raise ValueError("sequence_id values must be unique.")

print(f"✓ Accepted {len(user_df)} sequence(s).")
display(user_df)


In [ ]:
@tf.keras.utils.register_keras_serializable(package="pLM4CPP")
class MaskedAttentionPooling(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.attention_dense=tf.keras.layers.Dense(1,use_bias=True,name="residue_attention_score")
    def call(self, inputs):
        features,mask=inputs
        logits=tf.squeeze(self.attention_dense(features),axis=-1)
        mask=tf.cast(mask,logits.dtype)
        logits=logits+(1-mask)*tf.cast(-1e4,logits.dtype)
        weights=tf.nn.softmax(logits,axis=1)
        return tf.reduce_sum(features*tf.expand_dims(weights,-1),axis=1)
    def get_config(self):
        return super().get_config()

def forward_cpp_logit(model, emb, mask):
    x=tf.cast(emb,tf.float32)
    x=model.get_layer("embedding_layer_norm")(x,training=False)
    x=model.get_layer("residue_projection")(x,training=False)
    x=model.get_layer("residue_dropout")(x,training=False)
    x=model.get_layer("masked_attention_pooling")([x,mask],training=False)
    x=model.get_layer("peptide_dense_1")(x,training=False)
    x=model.get_layer("peptide_dropout_1")(x,training=False)
    x=model.get_layer("peptide_dense_2")(x,training=False)
    x=model.get_layer("peptide_dropout_2")(x,training=False)
    out=model.get_layer("cpp_probability")
    return tf.squeeze(tf.linalg.matmul(x,out.kernel)+out.bias,axis=-1)

def attention_scores(model,emb,mask):
    x=tf.cast(emb,tf.float32)
    x=model.get_layer("embedding_layer_norm")(x,training=False)
    x=model.get_layer("residue_projection")(x,training=False)
    x=model.get_layer("residue_dropout")(x,training=False)
    pool=model.get_layer("masked_attention_pooling")
    logits=tf.squeeze(pool.attention_dense(x,training=False),axis=-1)
    m=tf.cast(mask,logits.dtype)
    w=tf.nn.softmax(logits+(1-m)*tf.cast(-1e4,logits.dtype),axis=1)*m
    return tf.math.divide_no_nan(w,tf.reduce_sum(w,axis=1,keepdims=True)).numpy()

def grad_input_scores(model,emb_np,mask_np):
    emb=tf.convert_to_tensor(emb_np,dtype=tf.float32)
    mask=tf.convert_to_tensor(mask_np)
    with tf.GradientTape() as tape:
        tape.watch(emb)
        objective=tf.reduce_sum(forward_cpp_logit(model,emb,mask))
    grad=tape.gradient(objective,emb)
    score=tf.reduce_sum(tf.abs(grad*emb),axis=-1)*tf.cast(mask,tf.float32)
    return tf.math.divide_no_nan(score,tf.reduce_sum(score,axis=1,keepdims=True)).numpy()

def integrated_gradients_scores(model,emb_np,mask_np,steps=IG_STEPS):
    inp=tf.convert_to_tensor(emb_np,dtype=tf.float32)
    mask=tf.convert_to_tensor(mask_np)
    baseline=tf.zeros_like(inp); diff=inp-baseline; accum=tf.zeros_like(inp)
    for alpha in tf.linspace(0.0,1.0,steps):
        x=baseline+alpha*diff
        with tf.GradientTape() as tape:
            tape.watch(x)
            objective=tf.reduce_sum(forward_cpp_logit(model,x,mask))
        accum += tape.gradient(objective,x)
    ig=diff*(accum/tf.cast(steps,tf.float32))
    score=tf.reduce_sum(tf.abs(ig),axis=-1)*tf.cast(mask,tf.float32)
    return tf.math.divide_no_nan(score,tf.reduce_sum(score,axis=1,keepdims=True)).numpy()

def percentile_rank_valid(values,length):
    return pd.Series(np.asarray(values[:length],dtype=float)).rank(method="average",pct=True).to_numpy()


In [ ]:
def embed_esm(sequences,cfg):
    import esm
    loader=getattr(esm.pretrained,cfg["loader"])
    model,alphabet=loader()
    device="cuda" if torch.cuda.is_available() else "cpu"
    model=model.to(device).eval()
    converter=alphabet.get_batch_converter()
    X=np.zeros((len(sequences),MAX_LEN,cfg["dim"]),dtype=np.float32)
    M=np.zeros((len(sequences),MAX_LEN),dtype=np.uint8)

    for start in range(0,len(sequences),cfg["batch"]):
        bseq=sequences[start:start+cfg["batch"]]
        _,_,tokens=converter([(f"seq{i}",seq) for i,seq in enumerate(bseq)])
        tokens=tokens.to(device)
        with torch.no_grad():
            rep=model(tokens,repr_layers=[cfg["layer"]],return_contacts=False)["representations"][cfg["layer"]]
        for j,seq in enumerate(bseq):
            L=len(seq); X[start+j,:L]=rep[j,1:L+1].detach().cpu().float().numpy(); M[start+j,:L]=1
        del rep,tokens

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return X,M

def embed_prott5(sequences,cfg):
    from transformers import T5Tokenizer,T5EncoderModel
    tok=T5Tokenizer.from_pretrained(cfg["loader"],do_lower_case=False)
    model=T5EncoderModel.from_pretrained(cfg["loader"])
    device="cuda" if torch.cuda.is_available() else "cpu"
    model=model.to(device).eval()
    X=np.zeros((len(sequences),MAX_LEN,cfg["dim"]),dtype=np.float32)
    M=np.zeros((len(sequences),MAX_LEN),dtype=np.uint8)

    for i,seq in enumerate(sequences):
        enc=tok(" ".join(list(seq)),return_tensors="pt",add_special_tokens=True).to(device)
        with torch.no_grad():
            rep=model(**enc).last_hidden_state[0]
        L=len(seq); X[i,:L]=rep[:L].detach().cpu().float().numpy(); M[i,:L]=1

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return X,M


In [ ]:
sequences=user_df["sequence"].tolist()
predictions=user_df[["sequence_id","sequence"]].copy()
per_model_residue={}

for model_name in MODEL_NAMES:
    print("\n"+"="*70+"\n"+model_name)
    cfg=MODEL_CONFIGS[model_name]
    X,M=embed_esm(sequences,cfg) if cfg["type"]=="esm" else embed_prott5(sequences,cfg)

    clf=tf.keras.models.load_model(
        MODEL_ROOT/model_name/"final_attention_classifier.keras",
        custom_objects={"MaskedAttentionPooling":MaskedAttentionPooling},
        compile=False
    )
    threshold=float(json.loads((MODEL_ROOT/model_name/"selected_threshold.json").read_text())["selected_threshold"])

    prob=clf.predict([X,M],verbose=0).reshape(-1).astype(float)
    att=attention_scores(clf,X,M)
    gxi=grad_input_scores(clf,X,M)
    ig=integrated_gradients_scores(clf,X,M)

    predictions[f"{model_name}_probability"]=prob
    predictions[f"{model_name}_threshold"]=threshold
    predictions[f"{model_name}_prediction"]=(prob>=threshold).astype(int)

    rows=[]
    for i,row in user_df.iterrows():
        L=len(row.sequence)
        a=percentile_rank_valid(att[i],L)
        g=percentile_rank_valid(gxi[i],L)
        q=percentile_rank_valid(ig[i],L)
        grad_family=(g+q)/2
        score=(a+grad_family)/2
        rank=pd.Series(score).rank(method="average",pct=True).to_numpy()
        for pos,aa in enumerate(row.sequence,1):
            rows.append({
                "sequence_id":row.sequence_id,"sequence":row.sequence,
                "position":pos,"residue":aa,
                "attention_score":float(att[i,pos-1]),
                "gradient_input_score":float(gxi[i,pos-1]),
                "integrated_gradients_score":float(ig[i,pos-1]),
                "adjusted_model_consensus_rank":float(rank[pos-1]),
                "model_hotspot_top20":bool(rank[pos-1]>=0.80),
            })
    per_model_residue[model_name]=pd.DataFrame(rows)

    del clf,X,M,att,gxi,ig
    tf.keras.backend.clear_session(); gc.collect()

display(predictions)


In [ ]:
prob_cols=[f"{m}_probability" for m in MODEL_NAMES]
predictions["mean_ensemble_probability"]=predictions[prob_cols].mean(axis=1)
predictions["median_ensemble_probability"]=predictions[prob_cols].median(axis=1)
predictions["CPP_votes_out_of_4"]=predictions[[f"{m}_prediction" for m in MODEL_NAMES]].sum(axis=1)

identity=["sequence_id","sequence","position","residue"]
global_df=None
for model in MODEL_NAMES:
    part=per_model_residue[model][identity+["adjusted_model_consensus_rank","model_hotspot_top20"]].copy()
    part=part.rename(columns={
        "adjusted_model_consensus_rank":f"adjusted_rank_{model}",
        "model_hotspot_top20":f"hotspot_{model}",
    })
    global_df=part if global_df is None else global_df.merge(part,on=identity,validate="one_to_one")

rank_cols=[f"adjusted_rank_{m}" for m in MODEL_NAMES]
hot_cols=[f"hotspot_{m}" for m in MODEL_NAMES]
global_df["global_consensus_score"]=global_df[rank_cols].mean(axis=1)
global_df["global_consensus_rank"]=global_df.groupby("sequence_id")["global_consensus_score"].rank(method="average",pct=True)
global_df["consensus_hotspot_top20"]=global_df["global_consensus_rank"]>=0.80
global_df["PLM_hotspot_support"]=global_df[hot_cols].sum(axis=1).astype(int)
global_df["strict_consensus_hotspot_ge3PLMs"]=global_df["PLM_hotspot_support"]>=3
global_df["unanimous_hotspot_4PLMs"]=global_df["PLM_hotspot_support"]==4

predictions.to_csv(OUTPUT/"predictions.csv",index=False)
global_df.to_csv(OUTPUT/"consensus_residue_importance.csv",index=False)
for model,df in per_model_residue.items():
    df.to_csv(OUTPUT/f"{model}_residue_XAI.csv",index=False)

display(global_df.head(30))


In [ ]:
for sid,g in global_df.groupby("sequence_id",sort=False):
    g=g.sort_values("position")
    vals=g["global_consensus_rank"].to_numpy()
    residues=g["residue"].tolist()
    fig,ax=plt.subplots(figsize=(max(6,len(g)*0.34),1.8))
    im=ax.imshow(vals[np.newaxis,:],aspect="auto",vmin=0,vmax=1,cmap="viridis")
    ax.set_yticks([])
    ax.set_xticks(range(len(residues)))
    ax.set_xticklabels(residues,fontsize=10)
    ax.set_xlabel("Residue")
    ax.set_title(f"{sid} — pLM4CPP-XAI consensus importance")
    for j,row in enumerate(g.itertuples()):
        if row.consensus_hotspot_top20:
            ax.text(j,0,"★",ha="center",va="center",fontsize=10,color="white")
    cbar=fig.colorbar(im,ax=ax,fraction=0.025,pad=0.02)
    cbar.set_label("Within-sequence consensus rank")
    fig.tight_layout()
    fig.savefig(OUTPUT/f"{sid}_consensus_XAI.png",dpi=300,bbox_inches="tight")
    plt.close(fig)
print("✓ XAI heatmaps generated.")


In [ ]:
import shutil
archive=shutil.make_archive("/content/pLM4CPP_XAI_results","zip",OUTPUT)
print(archive)
files.download(archive)


## Interpretation

`predictions.csv` contains the four model probabilities, each model's published
validation-selected threshold/classification, the mean and median ensemble
probabilities, and the number of PLMs voting CPP.

`consensus_residue_importance.csv` contains residue-level consensus importance,
the top-20% consensus hotspot call, and the number of PLMs independently
supporting each residue as a hotspot.

These are computational model explanations rather than experimental proof of
causal membrane-translocation mechanisms.
